In [179]:
import pandas as pd
from urllib.parse import urlsplit
import sqlite3
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


In [2]:
conn = sqlite3.connect("../../db/telemetry.db")

In [181]:
df = pd.read_sql_query(
    """SELECT os.event_id, os.title, os.publisher, os.executable, os.event_start_time, os.event_end_time,
            browser.url, browser.title AS browser_tab_title, os.previous_events, class
        FROM os_events
        AS os LEFT JOIN browser_events AS browser
        ON os.event_id=browser.os_event_id
        WHERE class IS NOT NULL
        LIMIT 1000;""",
    conn,
    parse_dates = ["os_event_start_time", "os_event_end_time"]
)
df.head()

,event_id,title,publisher,executable,event_start_time,event_end_time,url,browser_tab_title,previous_events,class
0,e51b2531-129c-435c-ac8a-7115c3d53135,python,Python Software Foundation,python.exe,2026-09-27 20:18:21,2026-09-27 20:18:23,NaN,NaN,"[{""executable"": ""Code.exe"", ""title"": ""main.py ...",studying
1,fbd80d54-549c-4a1a-bec6-f82b13fdfab3,python,Python Software Foundation,python.exe,2026-09-27 20:18:23,2026-09-27 20:18:24,NaN,NaN,"[{""executable"": ""python.exe"", ""title"": ""python...",studying
2,627ac4f5-d842-417b-be55-b4f546f05e4f,python,Python Software Foundation,python.exe,2026-09-27 20:18:24,2026-09-27 20:18:28,NaN,NaN,"[{""executable"": ""python.exe"", ""title"": ""python...",studying
3,0a21a562-3d87-48fd-ad64-11a0aaa0b46e,python,Python Software Foundation,python.exe,2026-09-27 20:18:28,2026-09-27 20:21:13,NaN,NaN,"[{""executable"": ""python.exe"", ""title"": ""python...",studying
4,8d5134e9-5dd6-4887-9651-9f55deb68c2a,Notifications | LinkedIn - Google Chrome,Google LLC,chrome.exe,2026-09-27 20:21:13,2026-09-27 20:23:00,https://music.youtube.com/watch?v=QGJuMBdaqIw&...,Firework | YouTube Music,"[{""executable"": ""python.exe"", ""title"": ""python...",social_media


In [182]:
df[df["url"].notna()].head()

,event_id,title,publisher,executable,event_start_time,event_end_time,url,browser_tab_title,previous_events,class
4,8d5134e9-5dd6-4887-9651-9f55deb68c2a,Notifications | LinkedIn - Google Chrome,Google LLC,chrome.exe,2026-09-27 20:21:13,2026-09-27 20:23:00,https://music.youtube.com/watch?v=QGJuMBdaqIw&...,Firework | YouTube Music,"[{""executable"": ""python.exe"", ""title"": ""python...",social_media
5,8d5134e9-5dd6-4887-9651-9f55deb68c2a,Notifications | LinkedIn - Google Chrome,Google LLC,chrome.exe,2026-09-27 20:21:13,2026-09-27 20:23:00,https://music.youtube.com/watch?v=jUOdO5dqPeU&...,Firework | YouTube Music,"[{""executable"": ""python.exe"", ""title"": ""python...",social_media
6,8d5134e9-5dd6-4887-9651-9f55deb68c2a,Notifications | LinkedIn - Google Chrome,Google LLC,chrome.exe,2026-09-27 20:21:13,2026-09-27 20:23:00,https://www.linkedin.com/notifications/,Notifications | LinkedIn,"[{""executable"": ""python.exe"", ""title"": ""python...",social_media
7,8d5134e9-5dd6-4887-9651-9f55deb68c2a,Notifications | LinkedIn - Google Chrome,Google LLC,chrome.exe,2026-09-27 20:21:13,2026-09-27 20:23:00,https://www.linkedin.com/notifications/,Notifications | LinkedIn,"[{""executable"": ""python.exe"", ""title"": ""python...",social_media
9,4a230187-2ee1-4ab5-ac0d-95091b3e96cf,Fix Dataclass List Default - Google Chrome,Google LLC,chrome.exe,2026-09-27 20:23:02,2026-09-27 20:23:20,https://chatgpt.com/c/6ab92201-19f8-83ed-814b-...,Fix Dataclass List Default,"[{""executable"": ""Code.exe"", ""title"": ""main.py ...",studying


In [183]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 369 entries, 0 to 368
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   event_id           369 non-null    str  
 1   title              369 non-null    str  
 2   publisher          366 non-null    str  
 3   executable         369 non-null    str  
 4   event_start_time   369 non-null    str  
 5   event_end_time     369 non-null    str  
 6   url                211 non-null    str  
 7   browser_tab_title  211 non-null    str  
 8   previous_events    369 non-null    str  
 9   class              369 non-null    str  
dtypes: str(10)
memory usage: 29.0 KB


In [184]:
df.describe()

,event_id,title,publisher,executable,event_start_time,event_end_time,url,browser_tab_title,previous_events,class
count,369,369,366,369,369,369,211,211,369,369
unique,180,62,6,11,145,145,61,53,178,7
top,8d430e58-0c39-4ae9-ad2d-2b054d3f71aa,(5) The New Drone Causing Chaos Behind Russian...,Google LLC,chrome.exe,2026-09-27 21:59:08,2026-09-27 22:21:11,https://www.linkedin.com/feed/,Notifications | LinkedIn,"[{""executable"": ""Telegram.exe"", ""title"": ""\u20...",social_media
freq,47,47,215,215,47,47,22,25,47,104


In [185]:
df['executable'].value_counts()

executable
chrome.exe                  215
Telegram.exe                 55
steamwebhelper.exe           41
explorer.exe                 28
python.exe                    8
Ride-Win64-Shipping.exe       6
Code.exe                      5
ApplicationFrameHost.exe      5
tf_win64.exe                  3
ShellHost.exe                 2
SearchHost.exe                1
Name: count, dtype: int64

# Clean the data

In [162]:
df = df[~df['title'].isna()]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 314 entries, 0 to 313
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   event_id           314 non-null    str  
 1   title              314 non-null    str  
 2   publisher          311 non-null    str  
 3   executable         314 non-null    str  
 4   event_start_time   314 non-null    str  
 5   event_end_time     314 non-null    str  
 6   url                192 non-null    str  
 7   browser_tab_title  192 non-null    str  
 8   previous_events    314 non-null    str  
 9   class              314 non-null    str  
dtypes: str(10)
memory usage: 24.7 KB


In [163]:
df[(df['executable'] != 'chrome.exe') & (~df['url'].isna())]

,event_id,title,publisher,executable,event_start_time,event_end_time,url,browser_tab_title,previous_events,class


In [186]:
EXCLUDED_CLASSES_FOR_TRAINING = [
    'news'
]
df = df[~df['class'].isin(EXCLUDED_CLASSES_FOR_TRAINING)]

In [187]:
df[~df['url'].isna()]['executable'].value_counts()


executable
chrome.exe    164
Name: count, dtype: int64

In [188]:
df['url'] = np.where((df['url'].str.strip() == ''), 'chrome://newtab/', df['url'])
df['browser_tab_title'] = np.where((df['browser_tab_title'].str.strip() == ''), 'New tab', df['browser_tab_title'])
df.head()

,event_id,title,publisher,executable,event_start_time,event_end_time,url,browser_tab_title,previous_events,class
0,e51b2531-129c-435c-ac8a-7115c3d53135,python,Python Software Foundation,python.exe,2026-09-27 20:18:21,2026-09-27 20:18:23,NaN,NaN,"[{""executable"": ""Code.exe"", ""title"": ""main.py ...",studying
1,fbd80d54-549c-4a1a-bec6-f82b13fdfab3,python,Python Software Foundation,python.exe,2026-09-27 20:18:23,2026-09-27 20:18:24,NaN,NaN,"[{""executable"": ""python.exe"", ""title"": ""python...",studying
2,627ac4f5-d842-417b-be55-b4f546f05e4f,python,Python Software Foundation,python.exe,2026-09-27 20:18:24,2026-09-27 20:18:28,NaN,NaN,"[{""executable"": ""python.exe"", ""title"": ""python...",studying
3,0a21a562-3d87-48fd-ad64-11a0aaa0b46e,python,Python Software Foundation,python.exe,2026-09-27 20:18:28,2026-09-27 20:21:13,NaN,NaN,"[{""executable"": ""python.exe"", ""title"": ""python...",studying
4,8d5134e9-5dd6-4887-9651-9f55deb68c2a,Notifications | LinkedIn - Google Chrome,Google LLC,chrome.exe,2026-09-27 20:21:13,2026-09-27 20:23:00,https://music.youtube.com/watch?v=QGJuMBdaqIw&...,Firework | YouTube Music,"[{""executable"": ""python.exe"", ""title"": ""python...",social_media


# Feature Engineering

### Constants

In [189]:
BROWSER_EXECUTABLES = [
    'chrome.exe'
]

### Utility functions

In [190]:

def get_time_information(df):
    df['week_day'] = df["event_start_time"].dt.weekday
    df['hour'] = df["event_start_time"].dt.hour
    df["duration_seconds"] = (
        df["event_end_time"] - df["event_start_time"]
    ).dt.total_seconds()

    return df

def parse_dates(df):
    df['event_start_time'] = pd.to_datetime(df['event_start_time'], format='%Y-%m-%d %H:%M:%S')
    df['event_end_time'] = pd.to_datetime(df['event_end_time'], format='%Y-%m-%d %H:%M:%S')
    return df


In [191]:
df.head()

,event_id,title,publisher,executable,event_start_time,event_end_time,url,browser_tab_title,previous_events,class
0,e51b2531-129c-435c-ac8a-7115c3d53135,python,Python Software Foundation,python.exe,2026-09-27 20:18:21,2026-09-27 20:18:23,NaN,NaN,"[{""executable"": ""Code.exe"", ""title"": ""main.py ...",studying
1,fbd80d54-549c-4a1a-bec6-f82b13fdfab3,python,Python Software Foundation,python.exe,2026-09-27 20:18:23,2026-09-27 20:18:24,NaN,NaN,"[{""executable"": ""python.exe"", ""title"": ""python...",studying
2,627ac4f5-d842-417b-be55-b4f546f05e4f,python,Python Software Foundation,python.exe,2026-09-27 20:18:24,2026-09-27 20:18:28,NaN,NaN,"[{""executable"": ""python.exe"", ""title"": ""python...",studying
3,0a21a562-3d87-48fd-ad64-11a0aaa0b46e,python,Python Software Foundation,python.exe,2026-09-27 20:18:28,2026-09-27 20:21:13,NaN,NaN,"[{""executable"": ""python.exe"", ""title"": ""python...",studying
4,8d5134e9-5dd6-4887-9651-9f55deb68c2a,Notifications | LinkedIn - Google Chrome,Google LLC,chrome.exe,2026-09-27 20:21:13,2026-09-27 20:23:00,https://music.youtube.com/watch?v=QGJuMBdaqIw&...,Firework | YouTube Music,"[{""executable"": ""python.exe"", ""title"": ""python...",social_media


In [192]:
df = parse_dates(df)
df = get_time_information(df)
df["domain"] = df['url'].apply(lambda url: urlsplit(url).hostname if type(url)!=float else url) 
df['domain'].value_counts()

domain
www.youtube.com      80
www.linkedin.com     51
newtab               12
music.youtube.com     9
chatgpt.com           5
whats-new             5
github.com            2
Name: count, dtype: int64

In [193]:
df['is_os_event'] = df['executable'].isin(BROWSER_EXECUTABLES)
df = df.sort_values('is_os_event', ascending=False)
df.head()

,event_id,title,publisher,executable,event_start_time,event_end_time,url,browser_tab_title,previous_events,class,week_day,hour,duration_seconds,domain,is_os_event
246,49044959-1079-4619-9c0b-773cd6becb8b,New tab - Google Chrome,Google LLC,chrome.exe,2026-09-29 21:30:23,2026-09-29 21:31:57,https://www.youtube.com/watch?v=KND430-xHfg,(5) YouTube,"[{""executable"": ""steamwebhelper.exe"", ""title"":...",social_media,1,21,94.0,www.youtube.com,True
168,437ad122-ea99-41af-a312-f6e8ea22e98f,New tab - Google Chrome,Google LLC,chrome.exe,2026-09-28 22:22:51,2026-09-28 22:23:40,chrome://newtab/,New tab,"[{""executable"": ""explorer.exe"", ""title"": ""Task...",other,0,22,49.0,newtab,True
314,fda9556e-1a32-4b9a-a1ed-2ede508a9257,(5) шепіт нічного неба - YouTube - Google Chrome,Google LLC,chrome.exe,2026-09-30 22:56:51,2026-09-30 22:56:54,https://www.linkedin.com/notifications/,Notifications | LinkedIn,"[{""executable"": ""Telegram.exe"", ""title"": ""\u20...",social_media,2,22,3.0,www.linkedin.com,True
237,49044959-1079-4619-9c0b-773cd6becb8b,New tab - Google Chrome,Google LLC,chrome.exe,2026-09-29 21:30:23,2026-09-29 21:31:57,chrome://whats-new/?auto=true,What’s new,"[{""executable"": ""steamwebhelper.exe"", ""title"":...",social_media,1,21,94.0,whats-new,True
236,49044959-1079-4619-9c0b-773cd6becb8b,New tab - Google Chrome,Google LLC,chrome.exe,2026-09-29 21:30:23,2026-09-29 21:31:57,chrome://whats-new/,What’s new,"[{""executable"": ""steamwebhelper.exe"", ""title"":...",social_media,1,21,94.0,whats-new,True


In [194]:
import json
previous_events_df = df[['event_id', 'previous_events']]
previous_events_df['previous_events'] = previous_events_df['previous_events'].apply(json.loads)
previous_events_df = previous_events_df.explode('previous_events')
# display(previous_events_df['previous_events'].iloc[0])

# Extracting the columns from the previous events 
previous_events_df['previous_executable'] = previous_events_df['previous_events'].apply(lambda x: x.get('executable') if type(x)!=float else x)
previous_events_df['previous_publisher'] = previous_events_df['previous_events'].apply(lambda x: x.get('publisher') if type(x)!=float else x)
previous_events_df['event_start_time'] = previous_events_df['previous_events'].apply(lambda x: x.get('event_start_time') if type(x)!=float else x)
previous_events_df['event_end_time'] = previous_events_df['previous_events'].apply(lambda x: x.get('event_end_time') if type(x)!=float else x)


previous_events_df = parse_dates(previous_events_df)
previous_events_df = get_time_information(previous_events_df)

# previous_events_df['row_num'] = (
#     previous_events_df.groupby('event_id').cumcount() + 1
# )


# Aggregating the data
previous_events_df = (
    previous_events_df.groupby(
        'event_id'
    )
    .agg(
        previous_events_num_unique_executables=('previous_executable', 'nunique'),
        previous_events_num_unique_publishers=('previous_publisher', 'nunique'),
        previous_events_duration_seconds=('duration_seconds', 'sum'),
        
    )
    
)


previous_events_df.head()


,previous_events_num_unique_executables,previous_events_num_unique_publishers,previous_events_duration_seconds
event_id,,,
000ac24d-c6f0-4486-b0e8-1efc0f070c35,1,1,302.0
00f774f6-eabc-4eb6-9584-896208b09796,3,3,116.0
0128b82e-2d24-470e-aa38-4470a34cb82d,3,3,76.0
05a17acd-4c69-494b-b5bc-5e18479bf428,2,2,61.0
084eb0bb-3e9b-4c34-b4f7-24e6b5846a89,3,3,2935.0


In [195]:
df_flat = df
df_grouped = (
    df.groupby('event_id')
    .agg(
        title=('title', 'first'),
        executable=('executable', 'first'),
        publisher=('publisher', 'first'),
        event_start_time=('event_start_time', 'first'),
        event_end_time=('event_end_time', 'first'),
        previous_events=('previous_events', 'first'),
        week_day=('week_day', 'first'),
        hour=('hour', 'first'),
        num_unique_domains=('domain', 'nunique'),
        num_unique_urls=('url', 'nunique'),
        num_browser_events=('url','size'),
        duration_seconds=('duration_seconds', 'sum'),
        class_=('class', 'first')

    )
)
df_grouped.head()

,title,executable,publisher,event_start_time,event_end_time,previous_events,week_day,hour,num_unique_domains,num_unique_urls,num_browser_events,duration_seconds,class_
event_id,,,,,,,,,,,,,
000ac24d-c6f0-4486-b0e8-1efc0f070c35,TelegramDesktop,Telegram.exe,Telegram FZ-LLC,2026-09-30 22:56:36,2026-09-30 22:56:36,"[{""executable"": ""Telegram.exe"", ""title"": ""\u20...",2,22,0,0,1,0.0,social_media
00f774f6-eabc-4eb6-9584-896208b09796,Task Switching,explorer.exe,Microsoft Corporation,2026-09-30 22:58:56,2026-09-30 22:58:59,"[{""executable"": ""Telegram.exe"", ""title"": ""\u20...",2,22,0,0,1,3.0,other
0128b82e-2d24-470e-aa38-4470a34cb82d,Feed | LinkedIn - Google Chrome,chrome.exe,Google LLC,2026-09-27 20:33:18,2026-09-27 20:33:21,"[{""executable"": ""Code.exe"", ""title"": ""main.py ...",6,20,1,1,2,6.0,social_media
05a17acd-4c69-494b-b5bc-5e18479bf428,Media viewer,Telegram.exe,Telegram FZ-LLC,2026-09-30 23:02:23,2026-09-30 23:02:27,"[{""executable"": ""Telegram.exe"", ""title"": ""\u20...",2,23,0,0,1,4.0,other
084eb0bb-3e9b-4c34-b4f7-24e6b5846a89,Friends List,steamwebhelper.exe,Valve Corporation,2026-09-29 22:20:52,2026-09-29 22:20:54,"[{""executable"": ""Telegram.exe"", ""title"": ""(1) ...",1,22,0,0,1,2.0,gaming


In [196]:
df_grouped = df_grouped.merge(previous_events_df, on='event_id')

In [197]:
df_grouped['previous_events_parsed'] = df_grouped['previous_events'].apply(json.loads)
properties = ['executable', 'publisher']
for i in range(5):
    for property_ in properties:
        df_grouped[f'previous_event_{i + 1}_{property_}'] = df_grouped['previous_events_parsed'].apply(lambda arr: arr[i].get(property_) if len(arr) > i else None)
df_grouped.head()


,title,executable,publisher,event_start_time,event_end_time,previous_events,week_day,hour,num_unique_domains,num_unique_urls,...,previous_event_1_executable,previous_event_1_publisher,previous_event_2_executable,previous_event_2_publisher,previous_event_3_executable,previous_event_3_publisher,previous_event_4_executable,previous_event_4_publisher,previous_event_5_executable,previous_event_5_publisher
event_id,,,,,,,,,,,,,,,,,,,,,
000ac24d-c6f0-4486-b0e8-1efc0f070c35,TelegramDesktop,Telegram.exe,Telegram FZ-LLC,2026-09-30 22:56:36,2026-09-30 22:56:36,"[{""executable"": ""Telegram.exe"", ""title"": ""\u20...",2,22,0,0,...,Telegram.exe,Telegram FZ-LLC,Telegram.exe,Telegram FZ-LLC,Telegram.exe,Telegram FZ-LLC,Telegram.exe,Telegram FZ-LLC,Telegram.exe,Telegram FZ-LLC
00f774f6-eabc-4eb6-9584-896208b09796,Task Switching,explorer.exe,Microsoft Corporation,2026-09-30 22:58:56,2026-09-30 22:58:59,"[{""executable"": ""Telegram.exe"", ""title"": ""\u20...",2,22,0,0,...,Telegram.exe,Telegram FZ-LLC,explorer.exe,Microsoft Corporation,chrome.exe,Google LLC,Telegram.exe,Telegram FZ-LLC,chrome.exe,Google LLC
0128b82e-2d24-470e-aa38-4470a34cb82d,Feed | LinkedIn - Google Chrome,chrome.exe,Google LLC,2026-09-27 20:33:18,2026-09-27 20:33:21,"[{""executable"": ""Code.exe"", ""title"": ""main.py ...",6,20,1,1,...,Code.exe,Microsoft Corporation,python.exe,Python Software Foundation,chrome.exe,Google LLC,python.exe,Python Software Foundation,chrome.exe,Google LLC
05a17acd-4c69-494b-b5bc-5e18479bf428,Media viewer,Telegram.exe,Telegram FZ-LLC,2026-09-30 23:02:23,2026-09-30 23:02:27,"[{""executable"": ""Telegram.exe"", ""title"": ""\u20...",2,23,0,0,...,Telegram.exe,Telegram FZ-LLC,Telegram.exe,Telegram FZ-LLC,Telegram.exe,Telegram FZ-LLC,Telegram.exe,Telegram FZ-LLC,explorer.exe,Microsoft Corporation
084eb0bb-3e9b-4c34-b4f7-24e6b5846a89,Friends List,steamwebhelper.exe,Valve Corporation,2026-09-29 22:20:52,2026-09-29 22:20:54,"[{""executable"": ""Telegram.exe"", ""title"": ""(1) ...",1,22,0,0,...,Telegram.exe,Telegram FZ-LLC,steamwebhelper.exe,Valve Corporation,Telegram.exe,Telegram FZ-LLC,chrome.exe,Google LLC,Telegram.exe,Telegram FZ-LLC


In [198]:
df_grouped.info()

<class 'pandas.DataFrame'>
Index: 179 entries, 000ac24d-c6f0-4486-b0e8-1efc0f070c35 to ff81d290-ad51-4b27-8013-d1453c15236b
Data columns (total 27 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   title                                   179 non-null    str           
 1   executable                              179 non-null    str           
 2   publisher                               176 non-null    str           
 3   event_start_time                        179 non-null    datetime64[us]
 4   event_end_time                          179 non-null    datetime64[us]
 5   previous_events                         179 non-null    str           
 6   week_day                                179 non-null    int32         
 7   hour                                    179 non-null    int32         
 8   num_unique_domains                      179 non-null    int64         
 9   nu

In [199]:

text_features = ['title']
categorical_features = [
    'executable',
    'previous_event_1_executable',
    'previous_event_2_executable',
    'previous_event_3_executable',
    'previous_event_4_executable',
    'previous_event_5_executable',
    'previous_event_1_publisher',
    'previous_event_2_publisher',
    'previous_event_3_publisher',
    'previous_event_4_publisher',
    'previous_event_5_publisher',
    'week_day',
    'publisher'
]
numeric_features = [
    'hour', 
    'duration_seconds',
    'num_unique_domains',
    'num_unique_urls',
    'num_browser_events'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('title', TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2
        ), 'title'),
        ('categorical', OneHotEncoder(
            handle_unknown='ignore'
        ), categorical_features),
        ('numeric', Pipeline([
            # ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

In [200]:
X = df_grouped[
    text_features + categorical_features + numeric_features
]
y = df_grouped['class_']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=31,
    stratify=y
)


In [201]:
y.value_counts()

class_
social_media     56
gaming           50
other            44
studying         18
music             6
entertainment     5
Name: count, dtype: int64

In [202]:
model.fit(X, y)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](6,)","['entertainment','gaming','music','other','social_media','studying']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['title','executable','previous_event_1_executable',..., 'num_unique_domains','num_unique_urls','num_browser_events']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('title', ...), ('categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combine

In [216]:
# new_event = pd.DataFrame([{
#     'title': 'Visual Studio - Code',
#     'executable': 'code.exe',
#     'previous_event_1_executable': 'code.exe',
#     'week_day': 'Saturday',
#     'hour': 19,
#     'duration_seconds': 300,
#     'num_unique_urls': 0,
#     'previous_event_3_publisher': "code.exe",
#     'num_unique_domains': 0,
#     'previous_event_2_executable': "code.exe",
#     'previous_event_4_publisher': "Microsoft",
#     'previous_event_5_publisher': "Microsoft",
#     'previous_event_1_publisher': "Microsoft",
#     'previous_event_2_publisher': "Microsoft",
#     'num_browser_events': 0,
#     'previous_event_5_executable': "code.exe",
#     'previous_event_4_executable': "code.exe",
#     'previous_event_3_executable': "code.exe"
# }])

new_event = pd.DataFrame([{
    'title': 'Telegram',
    'executable': 'telegram.exe',
    'previous_event_1_executable': 'telegram.exe',
    'week_day': 'Saturday',
    'hour': 19,
    'duration_seconds': 300,
    'num_unique_urls': 0,
    'previous_event_3_publisher': "telegram.exe",
    'num_unique_domains': 0,
    'previous_event_2_executable': "telegram.exe",
    'previous_event_4_publisher': "Telegram FZ-LLC",
    'previous_event_5_publisher': "Telegram FZ-LLC",
    'previous_event_1_publisher': "Telegram FZ-LLC",
    'previous_event_2_publisher': "Telegram FZ-LLC",
    'num_browser_events': 0,
    'previous_event_5_executable': "telegram.exe",
    'previous_event_4_executable': "telegram.exe",
    'previous_event_3_executable': "telegram.exe",
    'publisher': 'Telegram FZ-LLC'
}])

prediction = model.predict(new_event)

print(prediction[0])

social_media
